<a href="https://colab.research.google.com/github/Ayoraham/receipt_scanner_V2/blob/main/Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install pytesseract

In [2]:
!pip install "Pillow>=10.0.0,<11.0.0" --upgrade

import PIL
print(f"Verified Pillow Version: {PIL.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 53.6 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
Verified Pillow Version: 11.3.0


In [1]:
import zipfile
from pathlib import Path
src_img_pth = "/content/drive/MyDrive/processed_data.zip"
dest_pth = "data"
with zipfile.ZipFile(src_img_pth,'r') as im_pth:
  im_pth.extractall(dest_pth)
print(f"All files extracted to {dest_pth}")

All files extracted to data


In [6]:
from transformers import LayoutLMv3Processor, LayoutLMv3ForSequenceClassification

In [3]:
from pathlib import Path
import os

In [4]:
data_pth = Path('data/content/screenshot_data')

In [ ]:
from google.colab.patches import cv2_imshow as show
img_pths = list(data_pth.glob("*/*"))
show(cv.imread(img_pths[0]))

In [7]:
processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base")
processor.image_processor.apply_ocr = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `LayoutLMv3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [8]:
conj = []
for sub in data_pth.iterdir():
  for subsub in sub.iterdir():
    conj.append(subsub)

### Quick Test

In [ ]:
import cv2 as cv
processor.image_processor.apply_ocr = True
img = cv.imread(conj[50])

In [23]:
encoding = processor(img,
                     max_length=512,
                     truncation=True,
                     padding="max_length",
                     return_tensors='pt')

In [27]:
model = LayoutLMv3ForSequenceClassification.from_pretrained('microsoft/layoutlmv3-base')

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3ForSequenceClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     | 
-----------------------------------+------------+-
layoutlmv3.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight         | MISSING    | 
classifier.dense.bias              | MISSING    | 
classifier.dense.weight            | MISSING    | 
classifier.out_proj.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [30]:
output = model(**encoding)


In [31]:
output

SequenceClassifierOutput(loss=None, logits=tensor([[-0.0314, -0.0202]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [35]:
!pip install -qqq torchmetrics
!pip install -qqq pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 19.1 MB/s eta 0:00:00


In [68]:
from torchmetrics import Accuracy
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split
import json


In [74]:
et_loc = "/content/drive/MyDrive/layoutlm_dataset.json"
with open(et_loc,'r') as file:
  extracted = json.load(file)

In [76]:
!gdown 1bQ4mFbVRUtOEJSe8b4hUYIcngSgfdldw

Downloading...
From (original): https://drive.google.com/uc?id=1bQ4mFbVRUtOEJSe8b4hUYIcngSgfdldw
From (redirected): https://drive.google.com/uc?id=1bQ4mFbVRUtOEJSe8b4hUYIcngSgfdldw&confirm=t&uuid=ece40493-f216-4f0c-9e1a-84ebf1a46862
To: /content/financial-documents-ocr.tar.xz
100% 147M/147M [00:03<00:00, 38.2MB/s]


In [78]:
!tar -xf financial-documents-ocr.tar.xz

In [ ]:
 img_pths = sorted(list(Path("images").glob("*/*.jpg")))
 print(len(img_pths))
 img_pths